In [1]:
!pip install ultralytics thop pandas


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# YOLOv8

In [27]:
import gc
import time
import random
import numpy as np
import torch
import pandas as pd

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from thop import profile

# ============================================================
# CONFIGURATION
# ============================================================

CHECKPOINTS = {
    "RGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLOv8n_weighted\weights\best.pt",
    "YGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLOv8n_weighted\weights\best.pt",
    "LGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLOv8n_weighted\weights\best.pt",
}

IMAGE_DIRS = {
    "RGB": Path(r"C:\Users\DELL\Desktop\c\SSL_RGB\test\images"),
    "YGB": Path(r"C:\Users\DELL\Desktop\c\SSL_YGB\test\images"),
    "LGB": Path(r"C:\Users\DELL\Desktop\c\SSL_LGB\test\images"),
}

IMG_SIZE = 640
WARMUP_ITERS = 30
BENCHMARK_REPEATS = 5
MAX_TEST_IMAGES = None

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
IS_CUDA = DEVICE.startswith("cuda")

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

print("Device:", DEVICE)
print("Precision: FP32")
print("Image size:", IMG_SIZE)

# ============================================================
# IMAGE DISCOVERY AND MATCHING
# ============================================================

def get_image_paths(folder):
    if not folder.is_dir():
        raise FileNotFoundError(f"Image directory not found: {folder}")

    paths = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    if not paths:
        raise RuntimeError(f"No images found in: {folder}")

    # Use relative filenames to match the same scenes.
    return {
        p.relative_to(folder).as_posix(): p
        for p in paths
    }


def get_matched_images():
    image_maps = {
        name: get_image_paths(folder)
        for name, folder in IMAGE_DIRS.items()
    }

    common_names = set.intersection(
        *(set(paths.keys()) for paths in image_maps.values())
    )

    if not common_names:
        raise RuntimeError(
            "No matching filenames across RGB, YGB, and LGB directories."
        )

    # Require identical test sets for a controlled comparison.
    for name, paths in image_maps.items():
        missing = set(common_names) - set(paths.keys())
        if missing:
            raise RuntimeError(f"Missing matched images in {name}.")

    counts = {name: len(paths) for name, paths in image_maps.items()}

    if len(set(counts.values())) != 1:
        print("Warning: representation directories have different image counts.")

    # Ensure every representation contains the same set of test scenes.
    sets = [set(paths.keys()) for paths in image_maps.values()]
    if not all(s == sets[0] for s in sets[1:]):
        raise RuntimeError(
            "The three directories do not contain identical filename sets. "
            "Use the same test scenes for a fair comparison."
        )

    selected = sorted(common_names)

    if MAX_TEST_IMAGES is not None:
        selected = selected[:MAX_TEST_IMAGES]

    print(f"Matched test images available: {len(common_names)}")
    print(f"Images used for benchmarking: {len(selected)}")

    return {
        name: [(key, image_maps[name][key]) for key in selected]
        for name in IMAGE_DIRS
    }


# ============================================================
# IMAGE PREPROCESSING
# ============================================================

letterbox = LetterBox(
    new_shape=(IMG_SIZE, IMG_SIZE),
    auto=False
)


def load_image_tensor(image_path):
    # Preserve the stored channel values of RGB, YGB, or LGB.
    with Image.open(image_path) as im:
        image = np.array(im.convert("RGB"))

    # Match YOLO's standard letterbox resizing and padding.
    image = letterbox(image=image)

    # HWC -> CHW, then normalize pixel values to [0, 1].
    image = np.ascontiguousarray(image.transpose(2, 0, 1))

    tensor = torch.from_numpy(image).float() / 255.0

    return tensor


# ============================================================
# PROFILE ONE CHECKPOINT
# ============================================================

def profile_checkpoint(name, checkpoint, image_items):
    checkpoint = Path(checkpoint)

    if not checkpoint.is_file():
        print(f"[SKIP] Checkpoint not found: {checkpoint}")
        return None

    print(f"\n{'=' * 75}")
    print(f"Profiling {name}")
    print(f"Checkpoint: {checkpoint}")
    print(f"Actual test images: {len(image_items)}")
    print("=" * 75, flush=True)

    yolo = YOLO(str(checkpoint))
    model = yolo.model.to(DEVICE).float().eval()

    total_params = sum(p.numel() for p in model.parameters())

    # Preprocess actual test images.
    # Keep them on CPU to avoid storing the entire test set in GPU memory.
    images = [
        (key, load_image_tensor(path))
        for key, path in image_items
    ]

    # FLOPs estimated using the first real test image.
    flops_g = float("nan")

    try:
        x = images[0][1].unsqueeze(0).to(DEVICE)

        with torch.inference_mode():
            macs, _ = profile(
                model,
                inputs=(x,),
                verbose=False
            )

        # THOP reports MACs; FLOPs conventionally estimated as 2 x MACs.
        flops_g = (2 * macs) / 1e9

        del x
        print(f"Estimated FLOPs/image: {flops_g:.4f} G", flush=True)

    except Exception as e:
        print(f"FLOPs profiling failed: {e}", flush=True)

    # Warm-up using real test images.
    with torch.inference_mode():
        for i in range(WARMUP_ITERS):
            idx = i % len(images)
            x = images[idx][1].unsqueeze(0).to(DEVICE)
            _ = model(x)

    if IS_CUDA:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        baseline_memory = torch.cuda.memory_allocated()

    # Randomized image order; repeat across the same image set.
    timings_ms = []

    for repeat in range(BENCHMARK_REPEATS):
        order = list(range(len(images)))
        random.Random(1000 + repeat).shuffle(order)

        with torch.inference_mode():
            for idx in order:
                # Transfer excluded from timed forward-pass latency.
                x = images[idx][1].unsqueeze(0).to(DEVICE)

                if IS_CUDA:
                    torch.cuda.synchronize()

                start = time.perf_counter()
                _ = model(x)

                if IS_CUDA:
                    torch.cuda.synchronize()

                timings_ms.append(
                    (time.perf_counter() - start) * 1000
                )

                del x

        print(
            f"[{name}] Repeat {repeat + 1}/{BENCHMARK_REPEATS} complete",
            flush=True
        )

    latency_ms = float(np.mean(timings_ms))
    latency_std_ms = float(np.std(timings_ms, ddof=1))
    throughput = 1000.0 / latency_ms

    if IS_CUDA:
        peak_memory_mb = (
            torch.cuda.max_memory_allocated() - baseline_memory
        ) / (1024 ** 2)
    else:
        peak_memory_mb = float("nan")

    result = {
        "Representation": name,
        "Images": len(images),
        "Parameters_M": total_params / 1e6,
        "FLOPs_G": flops_g,
        "Latency_ms_per_image": latency_ms,
        "Latency_SD_ms": latency_std_ms,
        "Throughput_images_per_sec": throughput,
        "Incremental_Peak_GPU_Memory_MB": peak_memory_mb,
    }

    del model, yolo, images
    gc.collect()

    if IS_CUDA:
        torch.cuda.empty_cache()

    return result


# ============================================================
# RUN ALL REPRESENTATIONS
# ============================================================

matched_images = get_matched_images()
all_results = []

for representation, checkpoint in CHECKPOINTS.items():
    result = profile_checkpoint(
        representation,
        checkpoint,
        matched_images[representation]
    )

    if result is not None:
        all_results.append(result)

# ============================================================
# DISPLAY RESULTS — NO CSV EXPORT
# ============================================================

if not all_results:
    raise RuntimeError(
        "No checkpoints were profiled. Verify checkpoint and image paths."
    )

df = pd.DataFrame(all_results)

print("\n" + "=" * 120)
print("YOLOv8n COMPUTATIONAL EFFICIENCY — REAL TEST IMAGES")
print("=" * 120)

columns = [
    "Representation",
    "Images",
    "Parameters_M",
    "FLOPs_G",
    "Latency_ms_per_image",
    "Latency_SD_ms",
    "Throughput_images_per_sec",
    "Incremental_Peak_GPU_Memory_MB",
]

print(df[columns].round(4).to_string(index=False))
print("\nProfiling completed. Results displayed in the terminal only.")

Device: cuda:0
Precision: FP32
Image size: 640
Matched test images available: 127
Images used for benchmarking: 127

Profiling RGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLOv8n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 8.1949 G
[RGB] Repeat 1/5 complete
[RGB] Repeat 2/5 complete
[RGB] Repeat 3/5 complete
[RGB] Repeat 4/5 complete
[RGB] Repeat 5/5 complete

Profiling YGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLOv8n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 8.1949 G
[YGB] Repeat 1/5 complete
[YGB] Repeat 2/5 complete
[YGB] Repeat 3/5 complete
[YGB] Repeat 4/5 complete
[YGB] Repeat 5/5 complete

Profiling LGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLOv8n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 8.1949 G
[LGB] Repeat 1/5 complete
[LGB] Repeat 2/5 complete
[LGB] Repeat 3/5 complete
[LGB] Repeat 4/5 complete
[LGB] Repeat 5/5 complete

# YOLOv12

In [28]:
import gc
import time
import random
import numpy as np
import torch
import pandas as pd

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from thop import profile

# ============================================================
# CONFIGURATION
# ============================================================

CHECKPOINTS = {
    "RGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLOv12n_weighted\weights\best.pt",
    "YGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLOv12n_weighted\weights\best.pt",
    "LGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLOv12n_weighted\weights\best.pt",
}

IMAGE_DIRS = {
    "RGB": Path(r"C:\Users\DELL\Desktop\c\SSL_RGB\test\images"),
    "YGB": Path(r"C:\Users\DELL\Desktop\c\SSL_YGB\test\images"),
    "LGB": Path(r"C:\Users\DELL\Desktop\c\SSL_LGB\test\images"),
}

IMG_SIZE = 640
WARMUP_ITERS = 30
BENCHMARK_REPEATS = 5
MAX_TEST_IMAGES = None

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
IS_CUDA = DEVICE.startswith("cuda")

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

print("Device:", DEVICE)
print("Precision: FP32")
print("Image size:", IMG_SIZE)

# ============================================================
# IMAGE DISCOVERY AND MATCHING
# ============================================================

def get_image_paths(folder):
    if not folder.is_dir():
        raise FileNotFoundError(f"Image directory not found: {folder}")

    paths = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    if not paths:
        raise RuntimeError(f"No images found in: {folder}")

    # Use relative filenames to match the same scenes.
    return {
        p.relative_to(folder).as_posix(): p
        for p in paths
    }


def get_matched_images():
    image_maps = {
        name: get_image_paths(folder)
        for name, folder in IMAGE_DIRS.items()
    }

    common_names = set.intersection(
        *(set(paths.keys()) for paths in image_maps.values())
    )

    if not common_names:
        raise RuntimeError(
            "No matching filenames across RGB, YGB, and LGB directories."
        )

    # Require identical test sets for a controlled comparison.
    for name, paths in image_maps.items():
        missing = set(common_names) - set(paths.keys())
        if missing:
            raise RuntimeError(f"Missing matched images in {name}.")

    counts = {name: len(paths) for name, paths in image_maps.items()}

    if len(set(counts.values())) != 1:
        print("Warning: representation directories have different image counts.")

    # Ensure every representation contains the same set of test scenes.
    sets = [set(paths.keys()) for paths in image_maps.values()]
    if not all(s == sets[0] for s in sets[1:]):
        raise RuntimeError(
            "The three directories do not contain identical filename sets. "
            "Use the same test scenes for a fair comparison."
        )

    selected = sorted(common_names)

    if MAX_TEST_IMAGES is not None:
        selected = selected[:MAX_TEST_IMAGES]

    print(f"Matched test images available: {len(common_names)}")
    print(f"Images used for benchmarking: {len(selected)}")

    return {
        name: [(key, image_maps[name][key]) for key in selected]
        for name in IMAGE_DIRS
    }


# ============================================================
# IMAGE PREPROCESSING
# ============================================================

letterbox = LetterBox(
    new_shape=(IMG_SIZE, IMG_SIZE),
    auto=False
)


def load_image_tensor(image_path):
    # Preserve the stored channel values of RGB, YGB, or LGB.
    with Image.open(image_path) as im:
        image = np.array(im.convert("RGB"))

    # Match YOLO's standard letterbox resizing and padding.
    image = letterbox(image=image)

    # HWC -> CHW, then normalize pixel values to [0, 1].
    image = np.ascontiguousarray(image.transpose(2, 0, 1))

    tensor = torch.from_numpy(image).float() / 255.0

    return tensor


# ============================================================
# PROFILE ONE CHECKPOINT
# ============================================================

def profile_checkpoint(name, checkpoint, image_items):
    checkpoint = Path(checkpoint)

    if not checkpoint.is_file():
        print(f"[SKIP] Checkpoint not found: {checkpoint}")
        return None

    print(f"\n{'=' * 75}")
    print(f"Profiling {name}")
    print(f"Checkpoint: {checkpoint}")
    print(f"Actual test images: {len(image_items)}")
    print("=" * 75, flush=True)

    yolo = YOLO(str(checkpoint))
    model = yolo.model.to(DEVICE).float().eval()

    total_params = sum(p.numel() for p in model.parameters())

    # Preprocess actual test images.
    # Keep them on CPU to avoid storing the entire test set in GPU memory.
    images = [
        (key, load_image_tensor(path))
        for key, path in image_items
    ]

    # FLOPs estimated using the first real test image.
    flops_g = float("nan")

    try:
        x = images[0][1].unsqueeze(0).to(DEVICE)

        with torch.inference_mode():
            macs, _ = profile(
                model,
                inputs=(x,),
                verbose=False
            )

        # THOP reports MACs; FLOPs conventionally estimated as 2 x MACs.
        flops_g = (2 * macs) / 1e9

        del x
        print(f"Estimated FLOPs/image: {flops_g:.4f} G", flush=True)

    except Exception as e:
        print(f"FLOPs profiling failed: {e}", flush=True)

    # Warm-up using real test images.
    with torch.inference_mode():
        for i in range(WARMUP_ITERS):
            idx = i % len(images)
            x = images[idx][1].unsqueeze(0).to(DEVICE)
            _ = model(x)

    if IS_CUDA:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        baseline_memory = torch.cuda.memory_allocated()

    # Randomized image order; repeat across the same image set.
    timings_ms = []

    for repeat in range(BENCHMARK_REPEATS):
        order = list(range(len(images)))
        random.Random(1000 + repeat).shuffle(order)

        with torch.inference_mode():
            for idx in order:
                # Transfer excluded from timed forward-pass latency.
                x = images[idx][1].unsqueeze(0).to(DEVICE)

                if IS_CUDA:
                    torch.cuda.synchronize()

                start = time.perf_counter()
                _ = model(x)

                if IS_CUDA:
                    torch.cuda.synchronize()

                timings_ms.append(
                    (time.perf_counter() - start) * 1000
                )

                del x

        print(
            f"[{name}] Repeat {repeat + 1}/{BENCHMARK_REPEATS} complete",
            flush=True
        )

    latency_ms = float(np.mean(timings_ms))
    latency_std_ms = float(np.std(timings_ms, ddof=1))
    throughput = 1000.0 / latency_ms

    if IS_CUDA:
        peak_memory_mb = (
            torch.cuda.max_memory_allocated() - baseline_memory
        ) / (1024 ** 2)
    else:
        peak_memory_mb = float("nan")

    result = {
        "Representation": name,
        "Images": len(images),
        "Parameters_M": total_params / 1e6,
        "FLOPs_G": flops_g,
        "Latency_ms_per_image": latency_ms,
        "Latency_SD_ms": latency_std_ms,
        "Throughput_images_per_sec": throughput,
        "Incremental_Peak_GPU_Memory_MB": peak_memory_mb,
    }

    del model, yolo, images
    gc.collect()

    if IS_CUDA:
        torch.cuda.empty_cache()

    return result


# ============================================================
# RUN ALL REPRESENTATIONS
# ============================================================

matched_images = get_matched_images()
all_results = []

for representation, checkpoint in CHECKPOINTS.items():
    result = profile_checkpoint(
        representation,
        checkpoint,
        matched_images[representation]
    )

    if result is not None:
        all_results.append(result)

# ============================================================
# DISPLAY RESULTS — NO CSV EXPORT
# ============================================================

if not all_results:
    raise RuntimeError(
        "No checkpoints were profiled. Verify checkpoint and image paths."
    )

df = pd.DataFrame(all_results)

print("\n" + "=" * 120)
print("YOLOv12n COMPUTATIONAL EFFICIENCY — REAL TEST IMAGES")
print("=" * 120)

columns = [
    "Representation",
    "Images",
    "Parameters_M",
    "FLOPs_G",
    "Latency_ms_per_image",
    "Latency_SD_ms",
    "Throughput_images_per_sec",
    "Incremental_Peak_GPU_Memory_MB",
]

print(df[columns].round(4).to_string(index=False))
print("\nProfiling completed. Results displayed in the terminal only.")

Device: cuda:0
Precision: FP32
Image size: 640
Matched test images available: 127
Images used for benchmarking: 127

Profiling RGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLOv12n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 7.4636 G
[RGB] Repeat 1/5 complete
[RGB] Repeat 2/5 complete
[RGB] Repeat 3/5 complete
[RGB] Repeat 4/5 complete
[RGB] Repeat 5/5 complete

Profiling YGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLOv12n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 7.4636 G
[YGB] Repeat 1/5 complete
[YGB] Repeat 2/5 complete
[YGB] Repeat 3/5 complete
[YGB] Repeat 4/5 complete
[YGB] Repeat 5/5 complete

Profiling LGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLOv12n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 7.4636 G
[LGB] Repeat 1/5 complete
[LGB] Repeat 2/5 complete
[LGB] Repeat 3/5 complete
[LGB] Repeat 4/5 complete
[LGB] Repeat 5/5 compl

# YOLOv26

In [29]:
import gc
import time
import random
import numpy as np
import torch
import pandas as pd

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from thop import profile

# ============================================================
# CONFIGURATION
# ============================================================

CHECKPOINTS = {
    "RGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLO26n_weighted\weights\best.pt",
    "YGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLO26n_weighted\weights\best.pt",
    "LGB": r"C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLO26n_weighted\weights\best.pt",
}

IMAGE_DIRS = {
    "RGB": Path(r"C:\Users\DELL\Desktop\c\SSL_RGB\test\images"),
    "YGB": Path(r"C:\Users\DELL\Desktop\c\SSL_YGB\test\images"),
    "LGB": Path(r"C:\Users\DELL\Desktop\c\SSL_LGB\test\images"),
}

IMG_SIZE = 640
WARMUP_ITERS = 30
BENCHMARK_REPEATS = 5
MAX_TEST_IMAGES = None

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
IS_CUDA = DEVICE.startswith("cuda")

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

print("Device:", DEVICE)
print("Precision: FP32")
print("Image size:", IMG_SIZE)

# ============================================================
# IMAGE DISCOVERY AND MATCHING
# ============================================================

def get_image_paths(folder):
    if not folder.is_dir():
        raise FileNotFoundError(f"Image directory not found: {folder}")

    paths = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    if not paths:
        raise RuntimeError(f"No images found in: {folder}")

    # Use relative filenames to match the same scenes.
    return {
        p.relative_to(folder).as_posix(): p
        for p in paths
    }


def get_matched_images():
    image_maps = {
        name: get_image_paths(folder)
        for name, folder in IMAGE_DIRS.items()
    }

    common_names = set.intersection(
        *(set(paths.keys()) for paths in image_maps.values())
    )

    if not common_names:
        raise RuntimeError(
            "No matching filenames across RGB, YGB, and LGB directories."
        )

    # Require identical test sets for a controlled comparison.
    for name, paths in image_maps.items():
        missing = set(common_names) - set(paths.keys())
        if missing:
            raise RuntimeError(f"Missing matched images in {name}.")

    counts = {name: len(paths) for name, paths in image_maps.items()}

    if len(set(counts.values())) != 1:
        print("Warning: representation directories have different image counts.")

    # Ensure every representation contains the same set of test scenes.
    sets = [set(paths.keys()) for paths in image_maps.values()]
    if not all(s == sets[0] for s in sets[1:]):
        raise RuntimeError(
            "The three directories do not contain identical filename sets. "
            "Use the same test scenes for a fair comparison."
        )

    selected = sorted(common_names)

    if MAX_TEST_IMAGES is not None:
        selected = selected[:MAX_TEST_IMAGES]

    print(f"Matched test images available: {len(common_names)}")
    print(f"Images used for benchmarking: {len(selected)}")

    return {
        name: [(key, image_maps[name][key]) for key in selected]
        for name in IMAGE_DIRS
    }


# ============================================================
# IMAGE PREPROCESSING
# ============================================================

letterbox = LetterBox(
    new_shape=(IMG_SIZE, IMG_SIZE),
    auto=False
)


def load_image_tensor(image_path):
    # Preserve the stored channel values of RGB, YGB, or LGB.
    with Image.open(image_path) as im:
        image = np.array(im.convert("RGB"))

    # Match YOLO's standard letterbox resizing and padding.
    image = letterbox(image=image)

    # HWC -> CHW, then normalize pixel values to [0, 1].
    image = np.ascontiguousarray(image.transpose(2, 0, 1))

    tensor = torch.from_numpy(image).float() / 255.0

    return tensor


# ============================================================
# PROFILE ONE CHECKPOINT
# ============================================================

def profile_checkpoint(name, checkpoint, image_items):
    checkpoint = Path(checkpoint)

    if not checkpoint.is_file():
        print(f"[SKIP] Checkpoint not found: {checkpoint}")
        return None

    print(f"\n{'=' * 75}")
    print(f"Profiling {name}")
    print(f"Checkpoint: {checkpoint}")
    print(f"Actual test images: {len(image_items)}")
    print("=" * 75, flush=True)

    yolo = YOLO(str(checkpoint))
    model = yolo.model.to(DEVICE).float().eval()

    total_params = sum(p.numel() for p in model.parameters())

    # Preprocess actual test images.
    # Keep them on CPU to avoid storing the entire test set in GPU memory.
    images = [
        (key, load_image_tensor(path))
        for key, path in image_items
    ]

    # FLOPs estimated using the first real test image.
    flops_g = float("nan")

    try:
        x = images[0][1].unsqueeze(0).to(DEVICE)

        with torch.inference_mode():
            macs, _ = profile(
                model,
                inputs=(x,),
                verbose=False
            )

        # THOP reports MACs; FLOPs conventionally estimated as 2 x MACs.
        flops_g = (2 * macs) / 1e9

        del x
        print(f"Estimated FLOPs/image: {flops_g:.4f} G", flush=True)

    except Exception as e:
        print(f"FLOPs profiling failed: {e}", flush=True)

    # Warm-up using real test images.
    with torch.inference_mode():
        for i in range(WARMUP_ITERS):
            idx = i % len(images)
            x = images[idx][1].unsqueeze(0).to(DEVICE)
            _ = model(x)

    if IS_CUDA:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        baseline_memory = torch.cuda.memory_allocated()

    # Randomized image order; repeat across the same image set.
    timings_ms = []

    for repeat in range(BENCHMARK_REPEATS):
        order = list(range(len(images)))
        random.Random(1000 + repeat).shuffle(order)

        with torch.inference_mode():
            for idx in order:
                # Transfer excluded from timed forward-pass latency.
                x = images[idx][1].unsqueeze(0).to(DEVICE)

                if IS_CUDA:
                    torch.cuda.synchronize()

                start = time.perf_counter()
                _ = model(x)

                if IS_CUDA:
                    torch.cuda.synchronize()

                timings_ms.append(
                    (time.perf_counter() - start) * 1000
                )

                del x

        print(
            f"[{name}] Repeat {repeat + 1}/{BENCHMARK_REPEATS} complete",
            flush=True
        )

    latency_ms = float(np.mean(timings_ms))
    latency_std_ms = float(np.std(timings_ms, ddof=1))
    throughput = 1000.0 / latency_ms

    if IS_CUDA:
        peak_memory_mb = (
            torch.cuda.max_memory_allocated() - baseline_memory
        ) / (1024 ** 2)
    else:
        peak_memory_mb = float("nan")

    result = {
        "Representation": name,
        "Images": len(images),
        "Parameters_M": total_params / 1e6,
        "FLOPs_G": flops_g,
        "Latency_ms_per_image": latency_ms,
        "Latency_SD_ms": latency_std_ms,
        "Throughput_images_per_sec": throughput,
        "Incremental_Peak_GPU_Memory_MB": peak_memory_mb,
    }

    del model, yolo, images
    gc.collect()

    if IS_CUDA:
        torch.cuda.empty_cache()

    return result


# ============================================================
# RUN ALL REPRESENTATIONS
# ============================================================

matched_images = get_matched_images()
all_results = []

for representation, checkpoint in CHECKPOINTS.items():
    result = profile_checkpoint(
        representation,
        checkpoint,
        matched_images[representation]
    )

    if result is not None:
        all_results.append(result)

# ============================================================
# DISPLAY RESULTS — NO CSV EXPORT
# ============================================================

if not all_results:
    raise RuntimeError(
        "No checkpoints were profiled. Verify checkpoint and image paths."
    )

df = pd.DataFrame(all_results)

print("\n" + "=" * 120)
print("YOLO26n COMPUTATIONAL EFFICIENCY — REAL TEST IMAGES")
print("=" * 120)

columns = [
    "Representation",
    "Images",
    "Parameters_M",
    "FLOPs_G",
    "Latency_ms_per_image",
    "Latency_SD_ms",
    "Throughput_images_per_sec",
    "Incremental_Peak_GPU_Memory_MB",
]

print(df[columns].round(4).to_string(index=False))
print("\nProfiling completed. Results displayed in the terminal only.")

Device: cuda:0
Precision: FP32
Image size: 640
Matched test images available: 127
Images used for benchmarking: 127

Profiling RGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLO26n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 5.8986 G
[RGB] Repeat 1/5 complete
[RGB] Repeat 2/5 complete
[RGB] Repeat 3/5 complete
[RGB] Repeat 4/5 complete
[RGB] Repeat 5/5 complete

Profiling YGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLO26n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 5.8986 G
[YGB] Repeat 1/5 complete
[YGB] Repeat 2/5 complete
[YGB] Repeat 3/5 complete
[YGB] Repeat 4/5 complete
[YGB] Repeat 5/5 complete

Profiling LGB
Checkpoint: C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLO26n_weighted\weights\best.pt
Actual test images: 127
Estimated FLOPs/image: 5.8986 G
[LGB] Repeat 1/5 complete
[LGB] Repeat 2/5 complete
[LGB] Repeat 3/5 complete
[LGB] Repeat 4/5 complete
[LGB] Repeat 5/5 complete

# Faster R-CNN

In [22]:
!pip install torch torchvision pandas fvcore


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_model():
    model = fasterrcnn_resnet50_fpn_v2(
        weights=None,
        weights_backbone=None,
        min_size=MIN_SIZE,
        max_size=MAX_SIZE,
        box_score_thresh=0.001,
        box_detections_per_img=300,
        image_mean=NORM_MEAN,
        image_std=NORM_STD,
    )
    
    state = torch.load(
        WEIGHTS_PATH,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(state)

    model.load_state_dict(state)

    if NEUTRAL_CONV1:
        make_stem_channel_neutral(model)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, NUM_CLASSES
    )

    assert list(model.transform.image_mean) == list(NORM_MEAN)
    assert list(model.transform.image_std) == list(NORM_STD)

    return model

In [31]:
from pathlib import Path
import torch

BASE_DIR = Path(r"C:\Users\DELL\Desktop\c")

MIN_SIZE, MAX_SIZE = 800, 1333
NEUTRAL_CONV1 = True
RUNS_DIR = BASE_DIR / "runs_frcnn_neutral"

WEIGHTS_DIR = Path(r"C:\Users\DELL\Desktop\c\runs\pretrained")
WEIGHTS_FILE = "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth"
WEIGHTS_PATH = WEIGHTS_DIR / WEIGHTS_FILE

NORM_MODE = "shared"

@torch.no_grad()
def make_stem_channel_neutral(model):
    """conv1 weight is [64, 3, 7, 7]. Replace the 3 input-channel filters by their mean, repeated 3 times.
    Response to a grey patch (R=G=B) is unchanged (3 * mean * g = sum(W_c) * g); colour-opponent filters (R-G ...) are removed,
    and every input channel now has the SAME filter, so no colour space is favoured at the first layer."""
    w = model.backbone.body.conv1.weight                    # nn.Parameter, stays trainable
    w.copy_(w.mean(dim=1, keepdim=True).repeat(1, w.shape[1], 1, 1))

if NORM_MODE == "shared":
    NORM_MEAN, NORM_STD = [0.449] * 3, [0.226] * 3
elif NORM_MODE == "none":
    NORM_MEAN, NORM_STD = [0.0] * 3, [1.0] * 3
else:
    raise ValueError(NORM_MODE)

WEIGHTS_URL = (
    "https://download.pytorch.org/models/"
    "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth"
)

NUM_CLASSES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"Pretrained weights not found: {WEIGHTS_PATH}")

print("Device:", DEVICE)
print("Pretrained weights:", WEIGHTS_PATH)
print("Weights exist:", WEIGHTS_PATH.is_file())

Device: cuda
Pretrained weights: C:\Users\DELL\Desktop\c\runs\pretrained\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
Weights exist: True


In [34]:
import gc
import time
import random
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from torchvision.transforms.functional import to_tensor

# ============================================================
# CONFIGURATION
# ============================================================

CHECKPOINTS = {
    "RGB": r"C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SSL_RGB_FasterRCNN_ce\best.pth",
    "YGB": r"C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SSL_YGB_FasterRCNN_ce\best.pth",
    "LGB": r"C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SSL_LGB_FasterRCNN_ce\best.pth",
}

# CHANGE THIS to your actual SSL test-image directory.
# This folder should contain the images for the RGB representation.
TEST_IMAGE_DIR = Path(r"C:\Users\DELL\Desktop\c\SSL_RGB\test\images")

# If each representation has its own test folder, specify them here.
IMAGE_DIRS = {
    "RGB": TEST_IMAGE_DIR,
    "YGB": Path(r"C:\Users\DELL\Desktop\c\SSL_YGB\test\images"),
    "LGB": Path(r"C:\Users\DELL\Desktop\c\SSL_LGB\test\images"),
}

WARMUP_ITERS = 10
BENCHMARK_REPEATS = 5
MAX_TEST_IMAGES = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# IMAGE DISCOVERY
# ============================================================

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def get_image_paths(folder):
    if not folder.is_dir():
        raise FileNotFoundError(f"Test-image directory not found: {folder}")

    paths = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    if not paths:
        raise RuntimeError(f"No images found in: {folder}")

    return paths

# ============================================================
# PROFILE ONE CHECKPOINT
# ============================================================

def profile_faster_rcnn(name, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    image_dir = IMAGE_DIRS[name]

    if not checkpoint_path.is_file():
        print(f"[SKIP] Checkpoint not found: {checkpoint_path}")
        return None

    paths = get_image_paths(image_dir)

    if MAX_TEST_IMAGES is not None:
        paths = paths[:MAX_TEST_IMAGES]
    print(f"\nProfiling {name}: {len(paths)} real test images", flush=True)

    model = build_model().to(DEVICE)

    state_dict = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True,
    )
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())

    # Load actual images, retaining their stored pixel values.
    images = []
    for path in paths:
        with Image.open(path) as im:
            image = to_tensor(im.convert("RGB"))
        images.append(image.to(DEVICE))

    # FLOPs estimate using the first real test image.
    flops_g = float("nan")
    try:
        from fvcore.nn import FlopCountAnalysis

        with torch.inference_mode():
            analysis = FlopCountAnalysis(model, ([images[0]],))
            analysis.unsupported_ops_warnings(False)
            analysis.uncalled_modules_warnings(False)
            flops_g = analysis.total() / 1e9

        print(f"Estimated FLOPs/image: {flops_g:.4f} G", flush=True)
    except Exception as e:
        print(f"FLOPs profiling unavailable: {e}", flush=True)

    # Warm up using a real test image.
    with torch.inference_mode():
        for _ in range(WARMUP_ITERS):
            _ = model([images[0]])

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        baseline_memory = torch.cuda.memory_allocated()

    # Randomize image order for each repeat to reduce order effects.
    timings_ms = []

    for repeat in range(BENCHMARK_REPEATS):
        order = list(range(len(images)))
        random.Random(1000 + repeat).shuffle(order)

        with torch.inference_mode():
            for idx in order:
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()

                start = time.perf_counter()
                _ = model([images[idx]])

                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()

                timings_ms.append((time.perf_counter() - start) * 1000)

        print(
            f"[{name}] Repeat {repeat + 1}/{BENCHMARK_REPEATS} complete",
            flush=True,
        )

    latency_ms = float(np.mean(timings_ms))
    latency_std_ms = float(np.std(timings_ms, ddof=1)) if len(timings_ms) > 1 else 0.0
    throughput = 1000.0 / latency_ms

    if DEVICE.type == "cuda":
        peak_memory_mb = (
            torch.cuda.max_memory_allocated() - baseline_memory
        ) / (1024 ** 2)
    else:
        peak_memory_mb = float("nan")

    result = {
        "Representation": name,
        "Images": len(images),
        "Parameters_M": total_params / 1e6,
        "FLOPs_G": flops_g,
        "Latency_ms_per_image": latency_ms,
        "Latency_SD_ms": latency_std_ms,
        "Throughput_images_per_sec": throughput,
        "Peak_GPU_Memory_MB": peak_memory_mb,
    }

    del model, state_dict, images
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

# ============================================================
# RUN
# ============================================================

all_results = []

for representation, checkpoint in CHECKPOINTS.items():
    result = profile_faster_rcnn(representation, checkpoint)
    if result is not None:
        all_results.append(result)

if not all_results:
    raise RuntimeError("No checkpoints were profiled. Check paths.")

df = pd.DataFrame(all_results)

print("\n" + "=" * 110)
print("FASTER R-CNN COMPUTATIONAL EFFICIENCY — REAL TEST IMAGES")
print("=" * 110)
print(df.round(4).to_string(index=False))
print("\nProfiling completed. Results displayed in terminal only.")

Using device: cuda

Profiling RGB: 127 real test images


C:\Users\DELL\miniconda3\Lib\site-packages\torch\nn\functional.py:4624: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\ops\boxes.py:167: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\ops\boxes.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_y = torch.min(boxes_y, torch.tensor(he

Estimated FLOPs/image: 322.4048 G


C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\models\detection\transform.py:308: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(s, dtype=torch.float32, device=boxes.device)
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\models\detection\transform.py:309: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  / torch.tensor(s_orig, dtype=torch.float32, device=boxes.device)


[RGB] Repeat 1/5 complete
[RGB] Repeat 2/5 complete
[RGB] Repeat 3/5 complete
[RGB] Repeat 4/5 complete
[RGB] Repeat 5/5 complete

Profiling YGB: 127 real test images


C:\Users\DELL\miniconda3\Lib\site-packages\torch\nn\functional.py:4624: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\ops\boxes.py:167: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\ops\boxes.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_y = torch.min(boxes_y, torch.tensor(he

... (3 outputs left out)

C:\Users\DELL\miniconda3\Lib\site-packages\torch\nn\functional.py:4624: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\ops\boxes.py:167: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\ops\boxes.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_y = torch.min(boxes_y, torch.tensor(he

Estimated FLOPs/image: 322.4048 G


C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\models\detection\transform.py:308: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(s, dtype=torch.float32, device=boxes.device)
C:\Users\DELL\miniconda3\Lib\site-packages\torchvision\models\detection\transform.py:309: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  / torch.tensor(s_orig, dtype=torch.float32, device=boxes.device)


[LGB] Repeat 1/5 complete
[LGB] Repeat 2/5 complete
[LGB] Repeat 3/5 complete
[LGB] Repeat 4/5 complete
[LGB] Repeat 5/5 complete

FASTER R-CNN COMPUTATIONAL EFFICIENCY — REAL TEST IMAGES
Representation  Images  Parameters_M  FLOPs_G  Latency_ms_per_image  Latency_SD_ms  Throughput_images_per_sec  Peak_GPU_Memory_MB
           RGB     127       43.2715 322.4048             2462.6008       219.9118                     0.4061            535.8037
           YGB     127       43.2715 322.4048             2419.4704       215.7563                     0.4133            535.8110
           LGB     127       43.2715 322.4048             2378.6399       210.1293                     0.4204            535.8096

Profiling completed. Results displayed in terminal only.
